# 01 - Exploratory Data Analysis

> Research prototype only. The dataset is simulated and labels are experimental proxies. No result here validates field deployment or early-warning lead time.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.data_loader import load_config, load_raw_data
from src.data_validation import validate_target

config = load_config()
frame = load_raw_data(config=config)
target = validate_target(frame, config)
print('shape:', frame.shape)

## 1. Schema, missing values and duplicates

In [ ]:
frame_info = pd.DataFrame({
    'dtype': frame.dtypes.astype(str),
    'missing': frame.isna().sum(),
    'unique': frame.nunique(),
})
frame_info

In [ ]:
print('Duplicate rows:', frame.duplicated().sum())
frame.describe().T

## 2. Target verification and class balance

`ActiveGL` is a continuous pit gain/loss indicator. The sign change from pit loss to pit gain coincides with the kick depth documented in the source paper. The resulting label is an experimental proxy, not authoritative ground truth.

In [ ]:
print('Rule:', target.rule)
print('Onset index:', target.onset_index, 'depth (ft):', target.onset_depth)
print('Positives:', target.n_positive, 'Negatives:', target.n_negative)
print('Warnings:', target.warnings)
ax = target.labels.value_counts().sort_index().plot(kind='bar', color='#4c78a8')
ax.set_xlabel('Proxy class'); ax.set_ylabel('Rows'); plt.show()

## 3. Feature distributions and correlation structure

In [ ]:
from src.feature_engineering import add_engineered_features
featured = add_engineered_features(frame, config)
featured[['FIn', 'FOut', 'Flow_Imbalance', 'DPPress', 'HLoad']].hist(bins=40, figsize=(12, 8))
plt.tight_layout(); plt.show()

In [ ]:
numeric = featured.select_dtypes(include='number')
plt.figure(figsize=(13, 10))
sns.heatmap(numeric.corr(), cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True)
plt.title('Pearson correlation (observation, not causation)')
plt.tight_layout(); plt.show()

## 4. Parameter behaviour by proxy class

In [ ]:
for column in ['FOut', 'Flow_Imbalance', 'DPPress', 'HLoad']:
    sns.boxplot(x=target.labels.to_numpy(), y=featured[column], showfliers=False)
    plt.xlabel('Proxy class (0=normal, 1=kick)')
    plt.title(column)
    plt.show()

## 5. Flow, pressure and pit-gain relationships

The identity `MRFlow = FIn - FOut` justifies `Flow_Imbalance = FOut - FIn`. A positive imbalance means return flow exceeds input flow, consistent with an influx.

In [ ]:
plt.scatter(featured['FIn'], featured['FOut'], c=target.labels, s=8, alpha=0.6, cmap='coolwarm')
plt.xlabel('FIn'); plt.ylabel('FOut'); plt.title('Flow in vs flow out'); plt.show()
plt.scatter(featured['DPPress'], featured['HLoad'], c=target.labels, s=8, alpha=0.6, cmap='coolwarm')
plt.xlabel('DPPress'); plt.ylabel('HLoad'); plt.title('Pressure vs hook load'); plt.show()

## 6. Reproduce published figures and report

In [ ]:
from src.eda import generate_figures, write_eda_report
figures = generate_figures(frame, target, config)
report = write_eda_report(frame, target, config, figures)
print(report)